In [1]:
partition = 478

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [8, 9, 10, 11, 12, 13]
hidden_dim = [1024, 768]
batch_size_values = [256, 512]
tree_feature_rates = [0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2]
lrs = [0.001, 0.01]

n_iter = 100
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0

sampled_configs = random.sample(param_space, min(n_iter, len(param_space)))
i = 1
for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")

    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
        
    i =i + 1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(best_acc)
print(f"Best accuracy: {best_acc}")



Running: n_tree=100, t_depth=11, hd=1024, batch_size=512, feature_rate=0.3, dropout=0.0, lr=0.001
1 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [17:25<00:00,  2.61s/it]



Best Accuracy: 0.556387

Running: n_tree=10, t_depth=10, hd=768, batch_size=512, feature_rate=0.3, dropout=0.2, lr=0.001
2 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [02:01<00:00,  3.30it/s]



Best Accuracy: 0.535429

Running: n_tree=100, t_depth=9, hd=1024, batch_size=512, feature_rate=0.4, dropout=0.2, lr=0.01
3 / 100
Use gtd478 dataset
Patience: 100


...


Best Accuracy: 0.584830

Running: n_tree=20, t_depth=9, hd=768, batch_size=256, feature_rate=0.3, dropout=0.1, lr=0.001
99 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [06:11<00:00,  1.08it/s]



Best Accuracy: 0.542914

Running: n_tree=50, t_depth=8, hd=1024, batch_size=512, feature_rate=0.1, dropout=0.2, lr=0.01
100 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [06:49<00:00,  1.02s/it]


Best Accuracy: 0.550399

Best hyperparameter configuration:
{'n_tree': 20, 'tree_depth': 13, 'batch_size': 256, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'lr': 0.01}
0.5853293413173652
Best accuracy: 0.5853293413173652


In [5]:


#{'n_tree': 10, 'tree_depth': 12, 'batch_size': 512, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'lr': 0.01}


In [6]:
"""

========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd478
  Hidden Dim: 768
  n_tree: 100, tree_depth: 12, tree_feature_rate: 0.3
  Batch size: 512, Dropout: 0.1, LR: 0.01

Best Accuracy: 0.5123
Weighted Precision: 0.5212, Recall: 0.5123, F1 Score: 0.4954, ROCAUC: 0.9268
Macro Precision: 0.5212, Recall: 0.5123, F1 Score: 0.4954, ROCAUC: 0.9268
Micro Precision: 0.5123, Recall: 0.5123, F1 Score: 0.5123, ROCAUC: 0.9334
"""

'\n\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd478\n  Hidden Dim: 768\n  n_tree: 100, tree_depth: 12, tree_feature_rate: 0.3\n  Batch size: 512, Dropout: 0.1, LR: 0.01\n\nBest Accuracy: 0.5123\nWeighted Precision: 0.5212, Recall: 0.5123, F1 Score: 0.4954, ROCAUC: 0.9268\nMacro Precision: 0.5212, Recall: 0.5123, F1 Score: 0.4954, ROCAUC: 0.9268\nMicro Precision: 0.5123, Recall: 0.5123, F1 Score: 0.5123, ROCAUC: 0.9334\n'

In [7]:
"""sys.argv = [
    'train.py',
    '-dataset', f'gtd{partition}',
    '-n_class', '30',
    '-gpuid', '0',
    '-n_tree', str(best_config['n_tree']),
    '-tree_depth', str(best_config['tree_depth']),
    '-batch_size', str(best_config['batch_size']),
    '-epochs', '1000',
    '-verbose', '1',
    '-jointly_training'
]"""

sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()


Use gtd478 dataset
Patience: 300


Training Epochs:   0%|          | 4/1500 [00:05<32:33,  1.31s/it]

Training Epochs:   3%|▎         | 50/1500 [01:02<30:24,  1.26s/it]

[Epoch 50] Train Loss: 1.3197, Eval Loss: 1.6449, Eval Accuracy: 0.5454


Training Epochs:   7%|▋         | 100/1500 [02:02<27:59,  1.20s/it]

...

[Epoch 1300] Train Loss: 1.1928, Eval Loss: 1.8574, Eval Accuracy: 0.5768


Training Epochs:  90%|█████████ | 1350/1500 [26:46<02:58,  1.19s/it]

[Epoch 1350] Train Loss: 1.1995, Eval Loss: 1.8629, Eval Accuracy: 0.5788


Training Epochs:  93%|█████████▎| 1391/1500 [27:36<02:09,  1.19s/it]

Early stopping at epoch 1392
Evaluating on test set with best model...


In [8]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.27      0.50      0.35       144
        African National Congress (South Africa)       0.52      0.87      0.65       144
                                Al-Qaida in Iraq       0.36      0.62      0.46       144
        Al-Qaida in the Arabian Peninsula (AQAP)       0.37      0.30      0.33       144
                                      Al-Shabaab       0.17      0.14      0.15       144
             Basque Fatherland and Freedom (ETA)       0.60      0.72      0.65       144
                                      Boko Haram       0.34      0.30      0.32       144
  Communist Party of India - Maoist (CPI-Maoist)       0.55      0.62      0.58       144
       Corsican National Liberation Front (FLNC)       0.57      0.74      0.64       144
                       Donetsk People's Republic       0.57      0.56      0.56       144
Farabundo

In [9]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [10]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true